In [2]:
!pip install deprecated
!pip install plotly

In [1]:
import os
import sys
from IPython.core.getipython import get_ipython

def search_folder(folder_name, start_path):
    for root, directories, files in os.walk(start_path):
        if folder_name in directories:
            return os.path.join(root, folder_name)
    return None

os.chdir('C:/Users/alvar/Desktop/ictus-synergies/scl_to_eurobench/')
sys.path.append('C:/Users/alvar/Desktop/ictus-synergies/conversion_stroke')
import txt_to_eurobench_utils as stroke_utils

sys.path.append('C:/Users/alvar/Desktop/common_eurobench/conversion')
import convert_utils
sys.path.append(search_folder(folder_name = 'emg-to-eurobench', start_path = '/'))
from preprocess_data_emg import preprocess_emg_eurobench


import pathlib as path
import pandas as pd
import importlib
os.chdir('C:/Users/alvar/Desktop/ictus-synergies/scl_to_eurobench/')

# Data loading

In [2]:
route_in = 'C:/Users/alvar/Desktop/Discrete_validation/Results'
route_out = 'C:/Users/alvar/Desktop/Discrete_validation/out_data'

if not os.path.exists(route_out):
    os.makedirs(route_out)

### Choose muscle configuration (only in this particular case where muscles depend on the subject)

In [3]:
def select_configuration(columns):
    dict_rename_cols_analog = {}
    dict_rename_cols_analog = {"Vastus medialis (mV)":"VaMe",
                                "Time (ms)":"time"}
    return dict_rename_cols_analog

In [50]:
importlib.reload(stroke_utils)
importlib.reload(convert_utils)
importlib.reload(preprocess_emg_eurobench)

sf = 1000
high_pass=20
low_pass=90
order=2 #3
low_pass_env=5
order_env=2
print(route_in)
if os.path.isdir(route_in):
    # Obtener la lista de carpetas dentro del directorio
    folders = [name for name in os.listdir(route_in) if os.path.isdir(os.path.join(route_in, name))]

    for folder in folders:
        print(folder)
        path_folder = route_in + '/' + folder
        # Define columns to keep and rename parameters (only for this particular case where each patient has a different thing)
        list_emg = os.listdir(path_folder)
        for file in list_emg:
            print(file)
            os.chdir(route_in + '/' + folder)
            raw_data = pd.read_csv(file, delimiter='\t', header=0)
            out_csv_name_txt = path.Path(route_in + '/' + folder).joinpath(path.Path(file).stem + '_clean.txt')
            raw_data.to_csv(out_csv_name_txt, sep = '\t', index= False)
            muscles = raw_data.columns.tolist()
            #dict_keep_drop_cols = {'EMG': ['Time', "GD", "GI", "GMD", "GMI", "CD", "CI", "ALD", "ALI", "ID", "II", "TAD", "TAI", "GTD", "GTI"]}
            dict_keep_drop_cols = {'EMG': ["Vastus medialis (mV)", 'Time (ms)']}
            dict_rename_cols = select_configuration(muscles)

            dfs = stroke_utils.convert_dir_txt_to_eurobench(path_folder, os.path.join(route_out, folder), '*_clean.txt', sf, dict_rename_cols=dict_rename_cols, dict_keep_drop_cols = dict_keep_drop_cols, b_frames=False, display = False, b_save_data=True)
                    
            # Filter and remove PLI  
            preprocess_emg_eurobench.preprocess_dir_emg_in_eurobench(route_out + '/' + folder, route_out + '/' + folder, sf, high_pass, low_pass, order, low_pass_env, order_env, display=False)


            

C:/Users/alvar/Desktop/Discrete_validation/Results
Patient_1
Stance_1.txt
------------------------------------
 File: C:\Users\alvar\Desktop\Discrete_validation\Results\Patient_1\Stance_1_clean.txt
------------------------------------
         Renaming columns using string vastus medialis (mv) to VaMe
         Renaming columns using string time (ms) to time
     ***** Saving at C:\Users\alvar\Desktop\Discrete_validation\out_data\Patient_1\Stance_1_clean_EMG.csv*****
     -----
------------------------------------
 File: C:\Users\alvar\Desktop\Discrete_validation\out_data\Patient_1\Stance_1_clean_EMG.csv
------------------------------------
------------------------------------
 File: C:\Users\alvar\Desktop\Discrete_validation\out_data\Patient_1\Stance_2_clean_EMG.csv
------------------------------------
------------------------------------
 File: C:\Users\alvar\Desktop\Discrete_validation\out_data\Patient_1\Stance_3_clean_EMG.csv
------------------------------------
--------------------

In [51]:
for folder in os.listdir(route_in):
    folder_route = os.path.join(route_in, folder)
    for file in os.listdir(folder_route):
        if '_clean' in file:
            os.remove(os.path.join(folder_route, file))